In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()

### nyse_daily.tsv
- exchange
- stock
- date
- open
- high
- low
- close
- volume
- adj_close

### nyse_dividends.tsv
- exchange
- stock
- date
- dividends

## Using Join

In [ ]:
nyse_daily_rdd = spark.sparkContext.textFile('/content/nyse_daily.tsv').map(lambda x: x.split('\t')).map(lambda x: ((x[0], x[1], x[2]), (float(x[6]) - float(x[3]))))
nyse_dividends_rdd = spark.sparkContext.textFile('/content/nyse_dividends.tsv').map(lambda x: x.split('\t')).map(lambda x: ((x[0], x[1], x[2]), float(x[3])))

In [46]:
nyse_daily_rdd.take(5)

[(('NYSE', 'CLI', '2009-12-31'), -0.8200000000000003),
 (('NYSE', 'CLI', '2009-12-30'), 0.17999999999999972),
 (('NYSE', 'CLI', '2009-12-29'), -0.3499999999999943),
 (('NYSE', 'CLI', '2009-12-28'), 0.01999999999999602),
 (('NYSE', 'CLI', '2009-12-24'), 0.0899999999999963)]

In [47]:
nyse_dividends_rdd.take(5)

[(('NYSE', 'CPO', '2009-12-30'), 0.14),
 (('NYSE', 'CPO', '2009-09-28'), 0.14),
 (('NYSE', 'CPO', '2009-06-26'), 0.14),
 (('NYSE', 'CPO', '2009-03-27'), 0.14),
 (('NYSE', 'CPO', '2009-01-06'), 0.14)]

In [50]:
nyse_daily_rdd.join(nyse_dividends_rdd).map(lambda x: (x[0][0], x[0][1], x[0][2], x[1][0], x[1][1])).take(5)

[('NYSE', 'CLI', '2009-10-01', -1.3499999999999979, 0.45),
 ('NYSE', 'CSL', '2009-08-12', -0.10999999999999943, 0.16),
 ('NYSE', 'CSL', '2009-02-18', 0.14000000000000057, 0.155),
 ('NYSE', 'CAT', '2009-01-15', 0.23000000000000398, 0.42),
 ('NYSE', 'CME', '2009-12-08', -0.6499999999999773, 1.15)]

## Using Broadcast Variable

In [51]:
nyse_daily_rdd = spark.sparkContext.textFile('/content/nyse_daily.tsv').map(lambda x: x.split('\t')). map(lambda x: ((x[0], x[1], x[2]), float(x[6]), float(x[3])))
nyse_dividends_rdd = spark.sparkContext.textFile('/content/nyse_dividends.tsv').map(lambda x: x.split('\t')).map(lambda x: ((x[0], x[1], x[2]), float(x[3])))

In [52]:
nyse_daily_rdd.take(5)

[(('NYSE', 'CLI', '2009-12-31'), 34.57, 35.39),
 (('NYSE', 'CLI', '2009-12-30'), 35.4, 35.22),
 (('NYSE', 'CLI', '2009-12-29'), 35.34, 35.69),
 (('NYSE', 'CLI', '2009-12-28'), 35.69, 35.67),
 (('NYSE', 'CLI', '2009-12-24'), 35.47, 35.38)]

In [53]:
nyse_dividends_rdd.take(5)

[(('NYSE', 'CPO', '2009-12-30'), 0.14),
 (('NYSE', 'CPO', '2009-09-28'), 0.14),
 (('NYSE', 'CPO', '2009-06-26'), 0.14),
 (('NYSE', 'CPO', '2009-03-27'), 0.14),
 (('NYSE', 'CPO', '2009-01-06'), 0.14)]

In [54]:
divident_dict = dict(nyse_dividends_rdd.collect())
divident_bdcst = spark.sparkContext.broadcast(divident_dict)
divident_bdcst

In [59]:
def broadcastJoin(pair_rdd):
  for pair in pair_rdd:
    key, close, open = pair
    dividend = divident_bdcst.value.get(key)
    if dividend is None:
      continue
    else:
      exchange, stock, date = key
      yield (exchange, stock, date, close - open, dividend)

In [61]:
result_rdd = nyse_daily_rdd.mapPartitions(broadcastJoin)
result_rdd.take(5)

[('NYSE', 'CLI', '2009-10-01', -1.3499999999999979, 0.45),
 ('NYSE', 'CLI', '2009-07-01', -0.14999999999999858, 0.45),
 ('NYSE', 'CLI', '2009-04-01', -0.389999999999997, 0.45),
 ('NYSE', 'CLI', '2009-01-02', -1.0999999999999979, 0.64),
 ('NYSE', 'CSL', '2009-11-12', -0.9500000000000028, 0.16)]